# Cross-Generator Transfer Matrix

**Cross-Generator Generalization in Deepfake Detection (IE7374)**

Amalgamates every `experiments/results/<run>.json` into the transfer matrix and the
seen-vs-unseen gap. Run this after teammates have committed their results JSONs
(ideally all eight runs, but it works with however many exist).

Each run wrote its rows in the shared schema (`docs/INTERFACES.md`, Contract 4), so
no manual bookkeeping is needed here.

## 1. Pull the latest results, then amalgamate
Make sure you have teammates' results first (`git pull`), then aggregate.

In [ ]:
!git pull --ff-only
!python experiments/transfer_matrix.py

## 2. Seen-vs-unseen gap (the headline)
Per run: mean AUC on the seen methods vs. the held-out (unseen) method, and the gap.

In [ ]:
import pandas as pd, os
p = "experiments/results/seen_vs_unseen_gap.csv"
if os.path.exists(p):
    g = pd.read_csv(p)
    display(g.round(3))
else:
    print("no results yet; run at least one evaluate.py")

## 3. Transfer-matrix heatmap (per backbone)
Rows = held-out method (trained on the other three), columns = tested-on method,
cell = AUC. The **diagonal** (held-out == tested-on) is the unseen cross-generator
number; off-diagonal cells are seen (in-distribution). The **SimSwap** column is the
self-generated generator, unseen for every run.

In [ ]:
import glob, pandas as pd, numpy as np, matplotlib.pyplot as plt

for f in sorted(glob.glob("experiments/results/transfer_matrix_*.csv")):
    bb = f.split("transfer_matrix_")[1].replace(".csv", "")
    m = pd.read_csv(f, index_col=0)
    fig, ax = plt.subplots(figsize=(1.3 * len(m.columns) + 2, 1.0 * len(m.index) + 2))
    im = ax.imshow(m.values, vmin=0.5, vmax=1.0, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(m.columns)))
    ax.set_xticklabels(m.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(m.index)))
    ax.set_yticklabels(m.index)
    ax.set_xlabel("tested on")
    ax.set_ylabel("held out (trained on the other 3)")
    ax.set_title(f"{bb}: cross-generator AUC")
    for i in range(len(m.index)):
        for j in range(len(m.columns)):
            v = m.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        color="white" if v < 0.8 else "black", fontsize=9)
    fig.colorbar(im, ax=ax, label="AUC")
    plt.tight_layout()
    out = f"experiments/results/transfer_matrix_{bb}.png"
    plt.savefig(out, dpi=120)
    print("saved", out)
    plt.show()

## What to read from it
- **Big diagonal drop** (held-out cell much lower than the off-diagonal seen cells)
  is the generalization gap, the project's headline finding.
- **Which held-out method drops most** tests the transfer-structure hypothesis
  (swap vs. reenactment, graphics vs. learned).
- **EfficientNet vs. XceptionNet** side by side answers whether the gap is
  architecture-specific or intrinsic to conventional CNNs.
- **SimSwap column** shows how a generator from outside FF++ (the one we generated
  ourselves) is detected, relative to the held-out FF++ methods.